In [1]:
BIBLIA_PATH = "../../data/processed/biblia/Ave Maria/Portugues-Catolica-AVM-All-Bible.txt"

In [2]:
from pprint import pprint


def get_metadata(line: str):
    """
    Get metadata for the line.
    Line must be in the format:
    <book> <chapter>:<verse> <text>
    """
    book = " ".join(line.split(":")[0].split(" ")[:-1])
    chapter = line.split(":")[0].split(" ")[-1]
    verse = line.split(":")[1].split(" ")[0]
    text = " ".join(line.split(":")[1].split(" ")[1:])
    return {"book": book, "chapter": chapter, "verse": verse, "text": text}


pprint(
    get_metadata(
        "Gênesis 4:21 O nome do seu irmão era Jubal, que foi o pai de todos aqueles que tocam a cítara e os instrumentos de sopro."
    )
)

pprint(
    get_metadata(
        "1 Samuel 1:1 Havia em Ramataim-Sofim um homem das montanhas de Efraim, chamado Elcana, filho de Jeroam, filho de Eliú, filho de Tou, filho de Suf, o efraimita."
    )
)

pprint(
    get_metadata(
        "2 Reis 19:17 É verdade, Senhor, que os reis da Assíria destruíram as nações e devastaram os seus territórios,"
    )
)

{'book': 'Gênesis',
 'chapter': '4',
 'text': 'O nome do seu irmão era Jubal, que foi o pai de todos aqueles que '
         'tocam a cítara e os instrumentos de sopro.',
 'verse': '21'}
{'book': '1 Samuel',
 'chapter': '1',
 'text': 'Havia em Ramataim-Sofim um homem das montanhas de Efraim, chamado '
         'Elcana, filho de Jeroam, filho de Eliú, filho de Tou, filho de Suf, '
         'o efraimita.',
 'verse': '1'}
{'book': '2 Reis',
 'chapter': '19',
 'text': 'É verdade, Senhor, que os reis da Assíria destruíram as nações e '
         'devastaram os seus territórios,',
 'verse': '17'}


In [3]:
from langchain.schema import Document

# 1. Carrega o conteúdo da Bíblia
with open(BIBLIA_PATH, "r", encoding="utf-8") as f:
    biblia_texto = f.readlines()

# 2. Cada linha vira um documento individual
docs = []
for i, line in enumerate(biblia_texto):
    if len(line) > 0:
        metadata = get_metadata(line)
        docs.append(
            Document(
                page_content=metadata["text"],
                metadata={
                    "book": metadata["book"],
                    "chapter": metadata["chapter"],
                    "verse": metadata["verse"],
                    "line_number": i,
                },
            )
        )


print(docs[9999])

page_content='É verdade, Senhor, que os reis da Assíria destruíram as nações e devastaram os seus territórios,
' metadata={'book': '2 Reis', 'chapter': '19', 'verse': '17', 'line_number': 9999}


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# 3. Inicializa o modelo de embedding
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

db = Chroma(
    collection_name="biblia",
    embedding_function=embedding_model,
    persist_directory="biblia_vectorstore",
    
)
# 4. Cria o índice vetorial
ids = db.add_documents(docs)


/tmp/ipykernel_79306/2295762258.py:9: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  db = Chroma(


['919ecaba-de8f-4caa-9dc1-20d7a985c796',
 'f2258d77-55a6-4766-9589-abf6c9427cb2',
 'f9e40e3d-f477-4fa8-ada8-b6973e0139b0',
 'fdd0dfad-2328-4e40-88be-dfa88dfa3188',
 '156f6e56-ff31-4b74-9c6a-805e02a9c40a',
 'e710b419-eda4-4c2f-b904-d3bcbb475d0d',
 '83a8a7c6-697b-4a31-8e64-746cb03c4398',
 '287459e9-9f59-4154-951a-281364e9bc9f',
 '6bd8dac1-3990-4733-be8d-764304fa7c3a',
 'f02b4f6f-a2fb-4659-a98f-40b902acd489',
 'ad4846df-8588-4991-bca4-2153f7cca599',
 '1050dd69-479c-4826-9585-066c6fcf7ddc',
 'bd31f877-072c-4e29-9509-cb873dd3dd11',
 '6b605f27-f1e2-4768-a4f2-cb68cd92b09e',
 '5bbee651-6101-457f-9efb-e0432be7f480',
 'c2ecd1d3-6740-40d1-948d-cb849f03529c',
 '696cf4f8-599e-4f22-a101-4147703717e7',
 '3914608e-d215-42ea-8fcc-e71d1887354c',
 '7ef10d62-e9b4-4803-ac6e-d8bdb745dc40',
 '3ff50807-34a4-4ac9-8a12-ba78dda96562',
 '9152cc3c-43e8-49f0-8637-5fa555a49fb7',
 '04935321-d10e-46c5-9a50-189076246622',
 '59f62e55-d7ba-4ef2-82e9-8b2d10af9327',
 'e02eea3b-324b-4d00-ab28-476db608de66',
 '4f2ce393-80db-

In [11]:
# Example query to search for similar documents
query = "No princípio, Deus criou o céu e a terra."
results = db.similarity_search(query, k=5)

# Print the results
for result in results:
    print(result)

page_content='No princípio, Deus criou o céu e a terra.
' metadata={'book': 'Gênesis', 'chapter': '1', 'line_number': 0, 'verse': '1'}
page_content='E chamou a fome sobre a terra, e os privou do pão que os sustentava.
' metadata={'book': 'Jó', 'chapter': '42', 'line_number': 16109, 'verse': '16'}
page_content='O Senhor me criou, como primícia de suas obras, desde o princípio, antes do começo da terra.
' metadata={'book': 'Provérbios', 'chapter': '8', 'line_number': 18595, 'verse': '22'}
page_content='No começo criastes a terra, e o céu é obra de vossas mãos.
' metadata={'book': 'Jó', 'chapter': '42', 'line_number': 16033, 'verse': '26'}
page_content='Por isso, o céu negou o seu orvalho e a terra, os seus frutos.
' metadata={'book': 'Ageu', 'chapter': '1', 'line_number': 27223, 'verse': '10'}
